In [1]:
import os
os.environ["HF_TOKEN"] = "your_token"

In [18]:
# =============================================================================  
# CELL 1: KAGGLE ENVIRONMENT SETUP  
# =============================================================================  
import os  
import sys  
import subprocess  
import warnings  
warnings.filterwarnings('ignore')  
  
# Install required packages  
!pip install -q sentence-transformers transformers matplotlib seaborn scikit-learn tqdm spacy  
  
# Kaggle-specific imports  
import pandas as pd  
import numpy as np  
import matplotlib.pyplot as plt  
import seaborn as sns  
from pathlib import Path  
import json  
import re  
import math  
import os
import spacy  
from collections import defaultdict, Counter  
from tqdm.auto import tqdm  
from sklearn.metrics.pairwise import cosine_similarity  
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline
os.environ['CUDA_LAUNCH_BLOCKING'] = "1"
# Set up Kaggle paths  
KAGGLE_INPUT_PATH = '/kaggle/input'  
KAGGLE_WORKING_PATH = '/kaggle/working'  
os.makedirs(KAGGLE_WORKING_PATH, exist_ok=True)  
  
# Your specific paths  
NER_MODEL_PATH = '/kaggle/input/models/tusharsharma2911/ner-model-finetuned/pytorch/default/1/legal_ner_trf_final/'  
DATASET_PATH = '/kaggle/input/datasets/tusharsharma2911/ramaaaa/legal-segmentation-100docs/'  
  
print(" Environment setup complete!")  
print(f" NER Model Path: {NER_MODEL_PATH}")  
print(f" Dataset Path: {DATASET_PATH}")

 Environment setup complete!
 NER Model Path: /kaggle/input/models/tusharsharma2911/ner-model-finetuned/pytorch/default/1/legal_ner_trf_final/
 Dataset Path: /kaggle/input/datasets/tusharsharma2911/ramaaaa/legal-segmentation-100docs/


In [22]:
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline
import torch
import os

class CustomLegalNER:
    def __init__(self, model_path, max_chunk_words=200):
        self.model_path = model_path
        self.nlp = None
        self.use_gpu = torch.cuda.is_available()
        self.max_chunk_words = max_chunk_words
        self._load_model()

    def _load_model(self):
        try:
            # Detect model type
            if os.path.exists(os.path.join(self.model_path, "meta.json")):
                import spacy
                if self.use_gpu:
                    spacy.require_gpu()
                self.nlp = spacy.load(self.model_path)
                print(" Loaded spaCy model")

            elif os.path.exists(os.path.join(self.model_path, "config.json")):
                # HuggingFace model
                tokenizer = AutoTokenizer.from_pretrained(self.model_path)
                model = AutoModelForTokenClassification.from_pretrained(self.model_path)

                device = 0 if self.use_gpu else -1

                self.nlp = pipeline(
                    "ner",
                    model=model,
                    tokenizer=tokenizer,
                    aggregation_strategy="simple",
                    device=device,
                    batch_size=16
                )

                print(f" Loaded HuggingFace model on {'GPU' if self.use_gpu else 'CPU'}")

            else:
                raise ValueError("Unknown model format")

        except Exception as e:
            print(f" Model loading failed: {e}")
            raise  # Fail loudly

    def _chunk_text(self, text):
        """Split long text into manageable chunks for NER"""
        words = text.split()
        chunks = []
        for i in range(0, len(words), self.max_chunk_words):
            chunks.append(" ".join(words[i:i + self.max_chunk_words]))
        return chunks

    def extract_entities(self, text):
        """Run NER safely on a single text"""
        if not self.nlp:
            return []

        all_entities = []

        try:
            chunks = self._chunk_text(text)
            for chunk in chunks:
                results = self.nlp(chunk)  #  do NOT pass truncation
                if isinstance(results, list):
                    for ent in results:
                        all_entities.append((
                            ent.get("word", ""),
                            ent.get("entity_group", ent.get("entity", "")),
                            round(ent.get("score", 0), 3)
                        ))
                elif hasattr(results, "ents"):
                    for ent in results.ents:
                        all_entities.append((ent.text, ent.label_))
            return all_entities

        except Exception as e:
            print(f" Error processing text: {e}")
            return []

    def process_sentences(self, sentences):
        """Process list of sentences and attach NER results"""
        for sent in sentences:
            try:
                sent['entities'] = self.extract_entities(sent['text'])
            except Exception as e:
                print(f" Sentence failed: {e}")
                sent['entities'] = []
        return sentences

In [ ]:
ner_model = CustomLegalNER(NER_MODEL_PATH)

In [23]:
# =============================================================================  
# CELL 3: DATA LOADING AND PREPROCESSING  
# =============================================================================  
def load_kaggle_dataset(dataset_path):  
    """Load dataset from Kaggle input directory"""  
    dataset_path = Path(dataset_path)  
      
    if not dataset_path.exists():  
        raise FileNotFoundError(f"Dataset {dataset_path} not found")  
      
    # Find all text files  
    files = list(dataset_path.glob("*.txt")) + list(dataset_path.glob("*.tsv"))  
    if not files:  
        # Try subdirectories  
        for subdir in dataset_path.iterdir():  
            if subdir.is_dir():  
                files.extend(list(subdir.glob("*.txt")))  
                files.extend(list(subdir.glob("*.tsv")))  
      
    documents = []  
      
    for file_path in files:  
        print(f"Loading: {file_path.name}")  
        try:  
            with open(file_path, 'r', encoding='utf-8') as f:  
                content = f.read().strip()  
                  
            if content:  
                # Simple sentence splitting (you can enhance this)  
                sentences = []  
                for i, line in enumerate(content.split('\n')):  
                    if line.strip():  
                        # Check if it's tab-separated format  
                        parts = line.strip().split('\t')  
                        if len(parts) >= 2:  
                            sentence, label = parts[0], parts[1]  
                        else:  
                            sentence, label = line.strip(), 'UNKNOWN'  
                          
                        sentences.append({  
                            'text': sentence,  
                            'label': label,  
                            'doc_id': file_path.stem,  
                            'line_num': i + 1  
                        })  
                  
                if sentences:  
                    documents.append({  
                        'doc_id': file_path.stem,  
                        'sentences': sentences,  
                        'source_file': str(file_path),  
                        'full_text': content  
                    })  
          
        except Exception as e:  
            print(f" Error loading {file_path}: {e}")  
      
    print(f" Loaded {len(documents)} documents")  
    return documents  
  
# Load the dataset  
documents = load_kaggle_dataset(DATASET_PATH)  
  
# Apply NER to all documents  
print(" Applying NER to documents...")  
for doc in tqdm(documents):  
    doc['sentences'] = ner_model.process_sentences(doc['sentences'])  
  
print(f" Processed {len(documents)} documents with NER")

Loading: BHC_BomHC_2007_1278.txt
Loading: SC_B Himmatlal Agrawal vs Competition Commission of India and Ors 18052018 SC.txt
Loading: SC_Excel Crop Care Limited vs Competition Commission of India and Ors 08052017 SC(1).txt
Loading: KHC_KolHC_1977_54.txt
Loading: CCI_Central_Organisation_for_Railway_Electrification_OCO2018060918171048137COM490335.txt
Loading: BHC_BomHC_2013_1012.txt
Loading: HC_Inox_Leisure_Limited_vs_Competition_Commission_of_DE2017271117162815102COM307750.txt
Loading: SC_2009_1357.txt
Loading: BHC_BomHC_2015_565.txt
Loading: CCI_Confederation_of_Real_Estate_Developers_AssociatioCO201807081816081635COM964576.txt
Loading: SC_2007_389.txt
Loading: SC_Brahm Dutt vs Union of India UOI 20012005 SC.txt
Loading: KHC_KolHC_1976_223.txt
Loading: CCI_Ashok_Kumar_Sharma_vs_Agni_Devices_Pvt_Ltd_0705201CO201516051522382724COM654807.txt
Loading: COM_TPM_Consultants_Pvt_Ltd_vs_Competition_Commission_TA2016110716153210127COM903224.txt
Loading: SC_2010_725.txt
Loading: SC_2010_116.txt
L

  0%|          | 0/100 [00:00<?, ?it/s]

 Processed 100 documents with NER


In [ ]:
# =============================================================================  
# CELL 4: LEGALPEGASUS EXTRACTIVE SUMMARIZATION  
# =============================================================================  
from transformers import PegasusForConditionalGeneration, PegasusTokenizer  
  
class LegalPegasusExtractiveSummarizer:  
    def __init__(self, model_name="t5-base"):  
        """Initialize LegalPegasus for extractive summarization"""  
        print(" Loading LegalPegasus for extractive summarization...")  
        self.tokenizer = PegasusTokenizer.from_pretrained(model_name)  
        self.model = PegasusForConditionalGeneration.from_pretrained(model_name)  
        self.redundancy_threshold = 0.3  
        self.coverage_threshold = 0.8  
        self.min_section_length = 50  
        print(" LegalPegasus extractive model loaded!")  
      
    def get_adaptive_summary_length(self, sent_count):  
        """Adaptive summary length based on document size"""  
        if sent_count <= 77:  
            const = 40.5421  
            slope = -0.2444  
        elif sent_count <= 122:  
            const = 29.5264  
            slope = -0.1013  
        else:  
            const = 17.8994  
            slope = -0.006  
          
        summary_percent = slope * sent_count + const  
        return max(10, summary_percent)  
      
    def trigram_blocking(self, sentences):  
        """Remove redundant sentences using trigram overlap"""  
        def get_ngrams(n, text):  
            words = text.lower().split()  
            ngrams = set()  
            for i in range(len(words) - n + 1):  
                ngrams.add(tuple(words[i:i+n]))  
            return ngrams  
          
        filtered_sentences = []  
        for sent in sentences:  
            sent_trigrams = get_ngrams(3, sent['text'])  
            is_redundant = False  
              
            for existing_sent in filtered_sentences:  
                existing_trigrams = get_ngrams(3, existing_sent['text'])  
                if len(sent_trigrams.intersection(existing_trigrams)) > 2:  
                    is_redundant = True  
                    break  
              
            if not is_redundant:  
                filtered_sentences.append(sent)  
          
        return filtered_sentences  
      
    def extractive_summarize_with_pegasus(self, document):  
        """Perform extractive summarization using LegalPegasus"""  
        sentences = document['sentences']  
          
        # Group sentences by rhetorical role for section-wise processing  
        sections = defaultdict(list)  
        for sent in sentences:  
            sections[sent['label']].append(sent)  
          
        selected_sentences = []  
        section_scores = {}  
          
        # Process each section with LegalPegasus  
        for section_label, section_sentences in sections.items():  
            # Combine sentences in this section  
            section_text = ' '.join([sent['text'] for sent in section_sentences])  
              
            # Generate extractive summary for this section  
            inputs = self.tokenizer(  
                section_text,   
                max_length=512,   
                truncation=True,   
                return_tensors="pt"  
            )  
              
            # Generate summary with extractive settings  
            summary_ids = self.model.generate(  
                inputs.input_ids,  
                max_length=200,  # Longer for extractive  
                min_length=50,  
                length_penalty=1.0,  
                num_beams=6,  
                early_stopping=True,  
                no_repeat_ngram_size=3  # Prevent repetition  
            )  
              
            summary = self.tokenizer.decode(  
                summary_ids[0],   
                skip_special_tokens=True  
            )  
              
            # Score sentences based on overlap with generated summary  
            section_sent_scores = []  
            for sent in section_sentences:  
                # Simple overlap scoring (you can use more sophisticated methods)  
                overlap = len(set(sent['text'].lower().split()) & set(summary.lower().split()))  
                score = overlap / len(sent['text'].split())  
                section_sent_scores.append((sent, score))  
              
            # Select top sentences from this section  
            section_sent_scores.sort(key=lambda x: x[1], reverse=True)  
            section_length = self.get_adaptive_summary_length(len(section_sentences))  
            num_to_select = max(1, int(len(section_sentences) * section_length / 100))  
              
            top_sentences = [item[0] for item in section_sent_scores[:num_to_select]]  
            selected_sentences.extend(top_sentences)  
            section_scores[section_label] = [item[1] for item in section_sent_scores[:num_to_select]]  
          
        # Apply trigram blocking across all selected sentences  
        selected_sentences = self.trigram_blocking(selected_sentences)  
          
        # Ensure mandatory categories are included  
        mandatory_labels = {'ISSUE', 'DECISION', 'RATIO'}  
        for sent in sentences:  
            if sent['label'] in mandatory_labels and sent not in selected_sentences:  
                selected_sentences.append(sent)  
          
        # Sort by original order  
        selected_sentences.sort(key=lambda x: x['line_num'])  
          
        # Calculate overall scores  
        all_scores = []  
        for sent in sentences:  
            if sent in selected_sentences:  
                if sent['label'] in section_scores:  
                    sent_score = max(section_scores[sent['label']]) if section_scores[sent['label']] else 0.5  
                else:  
                    sent_score = 0.5  
            else:  
                sent_score = 0.0  
            all_scores.append(sent_score)  
          
        return {  
            'extractive_summary': selected_sentences,  
            'scores': all_scores,  
            'summary_length_percent': self.get_adaptive_summary_length(len(sentences)),  
            'section_scores': section_scores  
        }  
  
# Initialize LegalPegasus extractive summarizer  
extractive_summarizer = LegalPegasusExtractiveSummarizer()  
  
# Apply LegalPegasus extractive summarization to all documents  
print(" Applying LegalPegasus extractive summarization...")  
for doc in tqdm(documents):  
    doc['extractive_result'] = extractive_summarizer.extractive_summarize_with_pegasus(doc)  
  
print(" LegalPegasus extractive summarization complete")

 Loading LegalPegasus for extractive summarization...


spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

You are using a model of type `t5` to instantiate a model of type `pegasus`. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.


...

Loading weights:   0%|          | 0/1 [00:00<?, ?it/s]

PegasusForConditionalGeneration LOAD REPORT from: t5-base
Key                                                                    | Status     | 
-----------------------------------------------------------------------+------------+-
decoder.block.{0...11}.layer.1.EncDecAttention.o.weight                | UNEXPECTED | 
decoder.block.{0...11}.layer.0.SelfAttention.o.weight                  | UNEXPECTED | 
decoder.block.{0...11}.layer.0.SelfAttention.v.weight                  | UNEXPECTED | 
decoder.block.{0...11}.layer.{0, 1, 2}.layer_norm.weight               | UNEXPECTED | 
encoder.block.{0...11}.layer.1.DenseReluDense.wo.weight                | UNEXPECTED | 
decoder.block.{0...11}.layer.1.EncDecAttention.v.weight                | UNEXPECTED | 
encoder.block.{0...11}.layer.0.SelfAttention.k.weight                  | UNEXPECTED | 
encoder.block.{0...11}.layer.0.SelfAttention.o.weight                  | UNEXPECTED | 
decoder.block.{0...11}.layer.2.DenseReluDense.wo.weight                |

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

 LegalPegasus extractive model loaded!
 Applying LegalPegasus extractive summarization...


  0%|          | 0/100 [00:00<?, ?it/s]

In [ ]:
# =============================================================================  
# CELL 5: ABSTRACTIVE SUMMARIZATION  
# =============================================================================  
from transformers import PegasusForConditionalGeneration, PegasusTokenizer  
  
class AbstractiveSummarizer:  
    def __init__(self, model_name="google/pegasus-xsum"):  
        """Initialize abstractive summarizer"""  
        print(" Loading Pegasus model...")  
        self.tokenizer = PegasusTokenizer.from_pretrained(model_name)  
        self.model = PegasusForConditionalGeneration.from_pretrained(model_name)  
        print(" Pegasus model loaded")  
      
    def summarize(self, extractive_result):  
        """Generate abstractive summary from extractive output"""  
        # Group sentences by rhetorical role  
        sections = defaultdict(list)  
        for sent in extractive_result['extractive_summary']:  
            sections[sent['label']].append(sent['text'])  
          
        abstractive_sections = {}  
          
        for section_label, sentences in sections.items():  
            # Combine sentences for each section  
            section_text = ' '.join(sentences)  
              
            # Generate summary  
            inputs = self.tokenizer(  
                section_text,   
                max_length=512,   
                truncation=True,   
                return_tensors="pt"  
            )  
              
            summary_ids = self.model.generate(  
                inputs.input_ids,  
                max_length=150,  
                min_length=30,  
                length_penalty=2.0,  
                num_beams=4,  
                early_stopping=True  
            )  
              
            summary = self.tokenizer.decode(  
                summary_ids[0],   
                skip_special_tokens=True  
            )  
              
            abstractive_sections[section_label] = summary  
          
        return abstractive_sections  
  
# Initialize abstractive summarizer  
abstractive_summarizer = AbstractiveSummarizer()  
  
# Apply abstractive summarization to all documents  
print(" Applying abstractive summarization...")  
for doc in tqdm(documents):  
    doc['abstractive_result'] = abstractive_summarizer.summarize(doc['extractive_result'])  
  
print(" Abstractive summarization complete")

In [ ]:
def get_pegasus_embeddings(texts, tokenizer, model, device):
    inputs = tokenizer(
        texts,
        truncation=True,
        padding=True,
        max_length=512,
        return_tensors="pt"
    )

    if device == "cuda":
        inputs = {k: v.cuda() for k, v in inputs.items()}
        model = model.cuda()

    with torch.no_grad():
        encoder = model.get_encoder()
        outputs = encoder(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            return_dict=True
        )

    embeddings = outputs.last_hidden_state.mean(dim=1)  # Mean pooling
    return embeddings.cpu().numpy()

In [ ]:
# =============================================================================  
# CELL 6: VALIDATION AND QUALITY METRICS  
# =============================================================================  
class ValidationMetrics:
    def __init__(self, tokenizer, model):
        self.redundancy_threshold = 0.3
        self.coverage_threshold = 0.8
        self.min_section_length = 50

        self.tokenizer = tokenizer
        self.model = model
        self.device = "cuda" if torch.cuda.is_available() else "cpu"

    def validate_extractive_summary(self, original_doc, extractive_result):
        original_sentences = original_doc['sentences']
        summary_sentences = extractive_result['extractive_summary']

        validation_metrics = {}

        # Coverage
        original_labels = {sent['label'] for sent in original_sentences}
        summary_labels = {sent['label'] for sent in summary_sentences}
        coverage = len(summary_labels.intersection(original_labels)) / max(1, len(original_labels))
        validation_metrics['label_coverage'] = coverage

        # Redundancy (FIXED)
        summary_texts = [sent['text'] for sent in summary_sentences]

        avg_similarity = 0  # default

        if len(summary_texts) > 1:
            try:
                embeddings = get_pegasus_embeddings(
                    summary_texts,
                    self.tokenizer,
                    self.model,
                    self.device
                )

                similarity_matrix = cosine_similarity(embeddings)
                np.fill_diagonal(similarity_matrix, 0)
                avg_similarity = np.mean(similarity_matrix)

            except Exception as e:
                print(f"Embedding error: {e}")
                avg_similarity = 0

        validation_metrics['redundancy_score'] = avg_similarity

        # Section length
        section_lengths = defaultdict(int)
        for sent in summary_sentences:
            section_lengths[sent['label']] += len(sent['text'].split())

        short_sections = [
            label for label, length in section_lengths.items()
            if length < self.min_section_length
        ]
        validation_metrics['short_sections'] = short_sections

        # Overall score (FIXED avg_similarity bug)
        quality_score = (
            coverage * 0.4 +
            (1 - avg_similarity) * 0.3 +
            (1 - len(short_sections) / max(1, len(section_lengths))) * 0.3
        )

        validation_metrics['overall_quality'] = quality_score

        return validation_metrics

    def validate_abstractive_summary(self, extractive_result, abstractive_result):
        validation_metrics = {}

        extractive_texts = [sent['text'] for sent in extractive_result['extractive_summary']]
        abstractive_texts = list(abstractive_result.values())

        if extractive_texts and abstractive_texts:
            try:
                extractive_emb = get_pegasus_embeddings(
                    extractive_texts,
                    self.tokenizer,
                    self.model,
                    self.device
                )

                abstractive_emb = get_pegasus_embeddings(
                    abstractive_texts,
                    self.tokenizer,
                    self.model,
                    self.device
                )

                similarities = cosine_similarity(
                    np.mean(extractive_emb, axis=0, keepdims=True),
                    abstractive_emb
                )

                validation_metrics['content_preservation'] = float(np.mean(similarities))

            except Exception as e:
                print(f"Embedding error: {e}")
                validation_metrics['content_preservation'] = 0

        # Length ratio
        extractive_length = sum(len(sent['text'].split()) for sent in extractive_result['extractive_summary'])
        abstractive_length = sum(len(text.split()) for text in abstractive_result.values())

        if extractive_length > 0:
            validation_metrics['length_ratio'] = abstractive_length / extractive_length

        # Coherence
        coherence_score = 0
        for section_text in abstractive_result.values():
            sentences = re.split(r'[.!?]+', section_text)
            complete = [s for s in sentences if len(s.strip()) > 10]
            coherence_score += len(complete) / max(1, len(sentences))

        validation_metrics['coherence_score'] = coherence_score / max(1, len(abstractive_result))

        return validation_metrics
  
print(" Validation complete")

In [ ]:
validator = ValidationMetrics(tokenizer, model)

In [ ]:
 =============================================================================  
# CELL 7: VISUALIZATION AND RESULTS  
# =============================================================================  
def generate_visualizations(documents):  
    """Generate validation visualizations"""  
    plt.style.use('seaborn-v0_8')  
    fig, axes = plt.subplots(2, 3, figsize=(20, 12))  
    fig.suptitle('Legal Summarization Pipeline - Validation Metrics', fontsize=16, fontweight='bold')  
      
    colors = ['#2E86AB', '#A23B72', '#F18F01', '#C73E1D', '#592E83', '#1B998B']  
      
    # 1. Extractive Quality Distribution  
    extractive_qualities = [doc['extractive_validation']['overall_quality'] for doc in documents]  
    axes[0, 0].hist(extractive_qualities, bins=15, alpha=0.8, color=colors[0], edgecolor='black')  
    axes[0, 0].set_title('Extractive Quality Distribution', fontweight='bold')  
    axes[0, 0].set_xlabel('Quality Score')  
    axes[0, 0].set_ylabel('Frequency')  
    axes[0, 0].grid(True, alpha=0.3)  
      
    # 2. Redundancy Scores  
    redundancy_scores = [doc['extractive_validation']['redundancy_score'] for doc in documents]  
    axes[0, 1].hist(redundancy_scores, bins=15, alpha=0.8, color=colors[1], edgecolor='black')  
    axes[0, 1].axvline(x=0.3, color='red', linestyle='--', linewidth=2, label='Threshold')  
    axes[0, 1].set_title('Redundancy Score Distribution', fontweight='bold')  
    axes[0, 1].set_xlabel('Redundancy Score')  
    axes[0, 1].set_ylabel('Frequency')  
    axes[0, 1].legend()  
    axes[0, 1].grid(True, alpha=0.3)  
      
    # 3. Label Coverage  
    coverage_scores = [doc['extractive_validation']['label_coverage'] for doc in documents]  
    axes[0, 2].hist(coverage_scores, bins=15, alpha=0.8, color=colors[2], edgecolor='black')  
    axes[0, 2].axvline(x=0.8, color='red', linestyle='--', linewidth=2, label='Threshold')  
    axes[0, 2].set_title('Label Coverage Distribution', fontweight='bold')  
    axes[0, 2].set_xlabel('Coverage Score')  
    axes[0, 2].set_ylabel('Frequency')  
    axes[0, 2].legend()  
    axes[0, 2].grid(True, alpha=0.3)  
      
    # 4. Content Preservation (Abstractive)  
    preservation_scores = [doc['abstractive_validation'].get('content_preservation', 0) for doc in documents]  
    axes[1, 0].hist(preservation_scores, bins=15, alpha=0.8, color=colors[3], edgecolor='black')  
    axes[1, 0].set_title('Content Preservation (Abstractive)', fontweight='bold')  
    axes[1, 0].set_xlabel('Preservation Score')  
    axes[1, 0].set_ylabel('Frequency')  
    axes[1, 0].grid(True, alpha=0.3)  
      
    # 5. Length Ratios  
    length_ratios = [doc['abstractive_validation'].get('length_ratio', 0) for doc in documents]  
    axes[1, 1].hist(length_ratios, bins=15, alpha=0.8, color=colors[4], edgecolor='black')  
    axes[1, 1].set_title('Abstractive/Extractive Length Ratio', fontweight='bold')  
    axes[1, 1].set_xlabel('Length Ratio')  
    axes[1, 1].set_ylabel('Frequency')  
    axes[1, 1].grid(True, alpha=0.3)  
      
    # 6. Overall Pipeline Performance  
    overall_scores = []  
    for doc in documents:  
        extractive_score = doc['extractive_validation']['overall_quality']  
        abstractive_score = doc['abstractive_validation'].get('coherence_score', 0)  
        overall_scores.append((extractive_score + abstractive_score) / 2)  
      
    axes[1, 2].hist(overall_scores, bins=15, alpha=0.8, color=colors[5], edgecolor='black')  
    axes[1, 2].set_title('Overall Pipeline Performance', fontweight='bold')  
    axes[1, 2].set_xlabel('Combined Score')  
    axes[1, 2].set_ylabel('Frequency')  
    axes[1, 2].grid(True, alpha=0.3)  
      
    plt.tight_layout()  
      
    # Save to Kaggle working directory  
    output_path = os.path.join(KAGGLE_WORKING_PATH, 'legal_summarization_metrics.png')  
    plt.savefig(output_path, dpi=300, bbox_inches='tight', facecolor='white')  
    plt.show()  
      
    return output_path  
  
def create_summary_results(documents):  
    """Create summary statistics and results DataFrame"""  
    results_data = []  
      
    for doc in documents:  
        doc_id = doc['doc_id']  
          
        # Extractive summary text  
        extractive_text = ' '.join([sent['text'] for sent in doc['extractive_result']['extractive_summary']])  
          
        # Abstractive summary text  
        abstractive_sections = doc['abstractive_result']  
        abstractive_text = ' | '.join([f"{section}: {text}" for section, text in abstractive_sections.items()])  
          
        results_data.append({  
            'doc_id': doc_id,  
            'extractive_summary': extractive_text,  
            'abstractive_summary': abstractive_text,  
            'extractive_quality': doc['extractive_validation']['overall_quality'],  
            'redundancy_score': doc['extractive_validation']['redundancy_score'],  
            'label_coverage': doc['extractive_validation']['label_coverage'],  
            'abstractive_coherence': doc['abstractive_validation']['coherence_score'],  
            'content_preservation': doc['abstractive_validation'].get('content_preservation', 0),  
            'length_ratio': doc['abstractive_validation'].get('length_ratio', 0)  
        })  
      
    # Create DataFrame  
    df = pd.DataFrame(results_data)  
      
    # Save results  
    csv_path = os.path.join(KAGGLE_WORKING_PATH, 'summarization_results.csv')  
    df.to_csv(csv_path, index=False)  
      
    # Save detailed JSON  
    json_path = os.path.join(KAGGLE_WORKING_PATH, 'detailed_results.json')  
    with open(json_path, 'w') as f:  
        json.dump(documents, f, indent=2)  
      
    print(f" Results saved to: {csv_path}")  
    print(f"📄 Detailed results saved to: {json_path}")  
      
    # Display summary statistics  
    print("\n Summary Statistics:")  
    print(f"Average Extractive Quality: {df['extractive_quality'].mean():.3f}")  
    print(f"Average Redundancy Score: {df['redundancy_score'].mean():.3f}")  
    print(f"Average Label Coverage: {df['label_coverage'].mean():.3f}")  
    print(f"Average Abstractive Coherence: {df['abstractive_coherence'].mean():.3f}")  
    print(f"Average Content Preservation: {df['content_preservation'].mean():.3f}")  
      
    return df, csv_path, json_path  
  
# Generate visualizations  
print(" Generating visualizations...")  
viz_path = generate_visualizations(documents)  
  
# Create summary results  
print(" Creating summary results...")  
results_df, csv_path, json_path = create_summary_results(documents)  
  
print(f" Pipeline complete! Visualizations saved to: {viz_path}")

In [ ]:
# =============================================================================  
# CELL 8: PIPELINE EXECUTION SUMMARY  
# =============================================================================  
def print_pipeline_summary(documents):  
    """Print comprehensive pipeline execution summary"""  
    print("=" * 80)  
    print("  LEGAL SUMMARIZATION PIPELINE - EXECUTION SUMMARY")  
    print("=" * 80)  
      
    total_docs = len(documents)  
    successful_docs = len([d for d in documents if d['extractive_validation']['overall_quality'] > 0.6])  
      
    print(f"\n DOCUMENT PROCESSING:")  
    print(f"   Total Documents: {total_docs}")  
    print(f"   Successfully Processed: {successful_docs}")  
    print(f"   Success Rate: {(successful_docs/total_docs)*100:.1f}%")  
      
    print(f"\n EXTRACTIVE SUMMARIZATION:")  
    avg_quality = np.mean([d['extractive_validation']['overall_quality'] for d in documents])  
    avg_redundancy = np.mean([d['extractive_validation']['redundancy_score'] for d in documents])  
    avg_coverage = np.mean([d['extractive_validation']['label_coverage'] for d in documents])  
      
    print(f"   Average Quality Score: {avg_quality:.3f}")  
    print(f"   Average Redundancy: {avg_redundancy:.3f}")  
    print(f"   Average Label Coverage: {avg_coverage:.3f}")  
      
    print(f"\n ABSTRACTIVE SUMMARIZATION:")  
    avg_coherence = np.mean([d['abstractive_validation']['coherence_score'] for d in documents])  
    avg_preservation = np.mean([d['abstractive_validation'].get('content_preservation', 0) for d in documents])  
    avg_length_ratio = np.mean([d['abstractive_validation'].get('length_ratio', 0) for d in documents])  
      
    print(f"   Average Coherence: {avg_coherence:.3f}")  
    print(f"   Average Content Preservation: {avg_preservation:.3f}")  
    print(f"   Average Length Ratio: {avg_length_ratio:.3f}")  
      
    print(f"\n OUTPUT FILES:")  
    print(f"   Visualizations: {viz_path}")  
    print(f"   Results CSV: {csv_path}")  
    print(f"   Detailed JSON: {json_path}")  
      
    print("\n" + "=" * 80)  
  
# Print summary  
print_pipeline_summary(documents)